In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('llm.env')
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I just found out about it?', 'Is it too late to start this course now?', 'Am I allowed to join after the course has already started?', 'If I join late, can I still get a certificate?', "What do I need to do to be eligible for the certificate if I'm joining now?"]


In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — is it still okay to start now, and if so, can I still get the certificate?', 'Can I join after the course has already started, or is that only for people who were there from the beginning?', 'If I enroll now, what do I need to do to still be eligible for the certificate?', 'Is late enrollment allowed in this course, and does it affect the certificate requirement?', 'I missed the start date — can I still take the course, and what happens with the project submission for certification?']


In [16]:
usage.input_tokens, usage.output_tokens

(207, 123)

In [17]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.0005535000000000001,
 'total_cost': 0.0007087500000000001}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — is it still okay to start now, and if so, can I still get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I join after the course has already started, or is that only for people who were there from the beginning?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll now, what do I need to do to still be eligible for the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Is late enrollment allowed in this course, and does it affect the certificate requirement?',
  'document': '74eb249bbf'},
 {'question': 'I missed the start date — can I still take the course, and what happens with the project submission for certification?',
  'document': '74eb249bbf'}]

In [21]:
import pandas as pd

In [22]:
pd.DataFrame(records)

,question,document
0,I just found this course late — is it still ok...,74eb249bbf
1,Can I join after the course has already starte...,74eb249bbf
2,"If I enroll now, what do I need to do to still...",74eb249bbf
3,"Is late enrollment allowed in this course, and...",74eb249bbf
4,I missed the start date — can I still take the...,74eb249bbf


In [23]:
from evaluation_utils import llm_structured_retry

In [24]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [25]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [29]:
ground_truth[1], usages[1]

({'question': 'Is it okay to start the course after it has already begun?',
  'document': '74eb249bbf'},
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=108, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=346))

In [30]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [31]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
results[1]

([{'question': 'I signed up for the LLM Zoomcamp — when should I get the confirmation email?',
   'document': '977bf7786c'},
  {'question': 'Do I actually need a registration confirmation before I can start the course?',
   'document': '977bf7786c'},
  {'question': 'Can I begin the lectures and homework right away even if I didn’t register?',
   'document': '977bf7786c'},
  {'question': 'Is the registration checked against some approved list, or can anyone submit homework while the form is open?',
   'document': '977bf7786c'},
  {'question': 'What’s the point of the course registration if it doesn’t affect access?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=337))

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [34]:
ground_truth[1], ground_truth[2]

({'question': 'If I start the course late, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get a certificate if I join after the course has started?',
  'document': '74eb249bbf'})

In [35]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07724774999999999

In [36]:
df_ground_truth = pd.DataFrame(ground_truth)


In [37]:
df_ground_truth

,question,document
0,Can I still join the course if I just found it...,74eb249bbf
1,"If I start the course late, am I still eligibl...",74eb249bbf
2,What do I need to do to get a certificate if I...,74eb249bbf
3,Is it okay to join the course now even though ...,74eb249bbf
4,Do I have to submit my project before submissi...,74eb249bbf
...,...,...
510,How do I install the exact requests version fr...,4b30b918bc
511,Why does pip keep pulling requests 2.28 when l...,4b30b918bc
512,What’s the easiest way to force-install reques...,4b30b918bc
513,I’m getting a 401 Client Error when using this...,4b30b918bc


In [39]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)